# Category 7 - TTS Voice Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares voice-output architectures for Ally Vision v2. The key question is whether speech is a separate post-processing service or part of the same realtime interaction loop.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
# Hardcoded from public-source URLs and project-grounded measurements only.
# No runtime web calls are performed in this notebook.
data = {
  "source_urls": {
    "Qwen pricing": "https://docs.qwencloud.com/developer-guides/getting-started/pricing",
    "OpenAI Realtime API": "https://openai.com/index/introducing-the-realtime-api/",
    "Google Cloud TTS docs": "https://cloud.google.com/text-to-speech/docs",
    "ElevenLabs pricing": "https://elevenlabs.io/pricing",
    "Project realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py"
  },
  "comparison_rows": [
    {
      "Metric": "Integrated same-session output",
      "Qwen Omni TTS (Tina)": "Yes, generated inside the realtime model session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "No, separate TTS layer from general model flow [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "No, separate API [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "No, separate API [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "No, separate API [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Streaming word-by-word audio",
      "Qwen Omni TTS (Tina)": "Yes [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "Streaming/bidirectional TTS documented in product family [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "Low-latency streaming positioning [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "Real-time TTS supported in product family [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "TTFA (public or proxy)",
      "Qwen Omni TTS (Tina)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "N/A (not publicly available) [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "N/A (not publicly available) [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "MOS / quality note",
      "Qwen Omni TTS (Tina)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "Established neural TTS family [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "Commercial high-quality TTS positioning [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "Established neural TTS family [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "Open-source quality varies by model [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "Expressive open-source baseline [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Number of voices",
      "Qwen Omni TTS (Tina)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "N/A (not publicly available) [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "N/A (not publicly available) [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Hindi voice quality",
      "Qwen Omni TTS (Tina)": "Project prompts target Hindi/Kannada/English flow [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "Broad language support family [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "N/A (not publicly available) [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "Hindi voices documented [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "Indian-language specialist positioning [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Kannada voice availability",
      "Qwen Omni TTS (Tina)": "Project path is multilingual; exact public Tina Kannada row not isolated [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "Broad language support family [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "N/A (not publicly available) [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "Kannada locales documented [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "Multilingual open-weight family [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "SSML support",
      "Qwen Omni TTS (Tina)": "N/A (not publicly available) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "Yes [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "Voice controls available through product APIs [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "Yes [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Emotion / prosody control",
      "Qwen Omni TTS (Tina)": "Prosody handled inside omni response generation [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "N/A (not publicly available) [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "Rich commercial voice controls [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "Commercial streaming TTS control [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "Expressive generation [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Price per 1M chars",
      "Qwen Omni TTS (Tina)": "Speech-to-speech token pricing in Qwen docs [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "See Cloud TTS pricing path [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "Commercial plan pricing page [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "See Azure Speech pricing path [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "N/A (not publicly available) [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "N/A (not publicly available) [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "N/A (not publicly available) [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "N/A (not publicly available) [source: https://github.com/suno-ai/bark]"
    },
    {
      "Metric": "Open-source / self-hostable",
      "Qwen Omni TTS (Tina)": "No [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
      "OpenAI TTS-1 / TTS-1-HD": "N/A (not publicly available) [source: https://openai.com/index/introducing-the-realtime-api/]",
      "Google Cloud TTS WaveNet": "N/A (not publicly available) [source: https://cloud.google.com/text-to-speech/docs]",
      "ElevenLabs Streaming": "N/A (not publicly available) [source: https://elevenlabs.io/pricing]",
      "Azure Neural TTS": "N/A (not publicly available) [source: https://learn.microsoft.com/en-us/azure/ai-services/speech-service/language-support?tabs=stt]",
      "Sarvam AI Bulbul": "N/A (not publicly available) [source: https://www.sarvam.ai/]",
      "Coqui TTS": "Yes [source: https://github.com/coqui-ai/TTS]",
      "Meta MMS TTS": "Yes [source: https://huggingface.co/facebook/mms-tts]",
      "PlayHT 2.0": "N/A (not publicly available) [source: https://play.ht/]",
      "Kokoro TTS": "Yes [source: https://huggingface.co/hexgrad/Kokoro-82M]",
      "Bark": "Yes [source: https://github.com/suno-ai/bark]"
    }
  ],
  "criteria": [
    "integrated loop",
    "streaming",
    "multilingual fit",
    "deployment flexibility"
  ],
  "fit_matrix": {
    "Qwen Omni TTS (Tina)": [
      1,
      1,
      1,
      0
    ],
    "OpenAI TTS-1 / TTS-1-HD": [
      0,
      1,
      1,
      0
    ],
    "Google Cloud TTS WaveNet": [
      0,
      1,
      1,
      0
    ],
    "ElevenLabs Streaming": [
      0,
      1,
      0.8,
      0
    ],
    "Azure Neural TTS": [
      0,
      1,
      1,
      0
    ],
    "Sarvam AI Bulbul": [
      0,
      0.6,
      1,
      0
    ],
    "Coqui TTS": [
      0,
      0.4,
      0.4,
      1
    ],
    "Meta MMS TTS": [
      0,
      0.4,
      1,
      1
    ],
    "PlayHT 2.0": [
      0,
      1,
      0.6,
      0
    ],
    "Kokoro TTS": [
      0,
      0.4,
      0.3,
      1
    ],
    "Bark": [
      0,
      0.4,
      0.3,
      1
    ]
  }
}


In [ ]:
import pandas as pd
df = pd.DataFrame(data["comparison_rows"])
display(df)


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
labels=list(data['fit_matrix'].keys())
values=[sum(v) for v in data['fit_matrix'].values()]
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 7 - TTS Voice Comparison Overall Fit Score")
ax.bar(labels,values,color=colors); ax.set_ylabel('Derived fit score',color='white'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.savefig(charts_dir / 'category7_tts_voice_comparison_chart1.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["Qwen Omni TTS (Tina)", "ElevenLabs Streaming", "Azure Neural TTS", "Google Cloud TTS WaveNet", "PlayHT 2.0"]; x=np.arange(len(criteria)); width=0.8/len(selected)
fig,ax=plt.subplots(figsize=(12,5)); style(ax,"Ally Vision v2 — Category 7 - TTS Voice Comparison Top-5 Criteria View")
for idx,label in enumerate(selected):
 vals=data['fit_matrix'][label]; color=ALLY if label==selected[0] else (DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else COMP); ax.bar(x+(idx-(len(selected)-1)/2)*width, vals, width=width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white'); plt.tight_layout(); plt.savefig(charts_dir / 'category7_tts_voice_comparison_chart2.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
ALLY="#4fc3f7"
COMP="#555555"
DEPR="#ff6b6b"
BG="#1a1a2e"
project_root=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir=project_root/'docs'/'comparisons'/'charts'
charts_dir.mkdir(parents=True,exist_ok=True)
colors=[]
for label in data['fit_matrix'].keys():
 colors.append(DEPR if ('deprecated' in label.lower() or 'shut down' in label.lower()) else (ALLY if label==list(data['fit_matrix'].keys())[0] else COMP))
def style(ax,title):
 ax.set_facecolor(BG); ax.figure.set_facecolor(BG); ax.set_title(title,color='white'); ax.tick_params(colors='white'); [s.set_color('#888888') for s in ax.spines.values()]; ax.grid(axis='y', color='#333333', alpha=0.3)
criteria=data['criteria']; selected=["Qwen Omni TTS (Tina)", "ElevenLabs Streaming", "Azure Neural TTS", "Google Cloud TTS WaveNet", "PlayHT 2.0"]; mat=np.array([data['fit_matrix'][k] for k in selected])
fig,ax=plt.subplots(figsize=(10,5)); ax.set_facecolor(BG); ax.figure.set_facecolor(BG); im=ax.imshow(mat,cmap='Blues',aspect='auto'); ax.set_title('Ally Vision v2 — Category 7 - TTS Voice Comparison Trade-off Heatmap',color='white'); ax.set_xticks(np.arange(len(criteria))); ax.set_xticklabels(criteria, rotation=20, ha='right', color='white'); ax.set_yticks(np.arange(len(selected))); ax.set_yticklabels(selected,color='white')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]): ax.text(j,i,f"{mat[i,j]:.0f}",ha='center',va='center',color='white')
plt.colorbar(im); plt.tight_layout(); plt.savefig(charts_dir / 'category7_tts_voice_comparison_chart3.png',dpi=150,bbox_inches='tight'); plt.show()


## Data Sources

| # | Source Name | URL | Value extracted |
|---|-------------|-----|-----------------|
| 1 | Qwen pricing | https://docs.qwencloud.com/developer-guides/getting-started/pricing | speech-to-speech pricing context |
| 2 | OpenAI Realtime API | https://openai.com/index/introducing-the-realtime-api/ | stitched TTS vs realtime speech path |
| 3 | Google Cloud TTS docs | https://cloud.google.com/text-to-speech/docs | streaming and language support family |
| 4 | ElevenLabs pricing | https://elevenlabs.io/pricing | streaming product/pricing notes |
| 5 | Project realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | current Tina-based realtime path |


## CONCLUSION

Qwen Omni wins because Ally Vision v2 does not have to turn one response into two separate networked phases. Speech output is part of the same realtime interaction loop, which keeps interruption handling and operational complexity lower than a stitched LLM-plus-TTS architecture.

→ Chosen for Ally Vision v2 ✅
